# OMI Transactions Exploration

Exploratory analysis of OMI normalized transaction volumes (NTN) for Italian municipalities, 2011–2025.

**Workflow:** ingestion → schema harmonization → key validation → controlled joins → residential reconciliation → national/regional trends → size mix → municipality panel → analytical hand-off to price/volume analysis.


## 1. Setup

The loader discovers annual releases instead of hard-coding years. Each release contains `LISTA-COM`, `VALORI-RES`, `VALORI-COM` and `VALORI-PER`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
TRANSACTIONS_FOLDER = PROJECT_ROOT / 'data' / 'raw' / 'transactions'
YEAR_FOLDERS = sorted(p for p in TRANSACTIONS_FOLDER.iterdir() if p.is_dir() and p.name.isdigit())
YEARS = [int(p.name) for p in YEAR_FOLDERS]
print(f'Releases: {min(YEARS)}–{max(YEARS)} ({len(YEARS)} years)')

## 2. Load and harmonize annual releases

OMI uses semicolon-separated files and comma decimals. Year-specific field names are normalized so the analysis is independent of the release year.


In [ ]:
def get_files(folder):
    files = {f.name.lower(): f for f in folder.iterdir() if f.is_file()}
    patterns = {'lista_com':'lista-com','valori_res':'valori-res','valori_com':'valori-com','valori_per':'valori-per'}
    out = {}
    for key, pattern in patterns.items():
        matches = [f for name, f in files.items() if pattern in name]
        if len(matches) != 1:
            raise ValueError(f'{folder.name}: expected one {pattern} file, found {len(matches)}')
        out[key] = matches[0]
    return out

def load_source(path, year):
    df = pd.read_csv(path, sep=';', decimal=',')
    df.columns = [str(c).strip() for c in df.columns]
    codcom = next((c for c in df.columns if re.search(r'codcom$', c, re.I)), None)
    if codcom is None:
        raise KeyError(f'{path.name}: CodCom column not found')
    df = df.rename(columns={codcom: 'CodCom'})
    df.columns = [re.sub(str(year), 'YEAR', c, flags=re.I) for c in df.columns]
    return df

sources = {k:{} for k in ['lista_com','valori_res','valori_com','valori_per']}
inventory = []
for folder in YEAR_FOLDERS:
    year = int(folder.name)
    files = get_files(folder)
    inventory.append({'year':year, **{k:v.name for k,v in files.items()}})
    for key, path in files.items():
        sources[key][year] = load_source(path, year)

inventory_df = pd.DataFrame(inventory)
inventory_df

## 3. Validate keys before joining

`CodCom` is the municipality key. A duplicate key would make a later merge potentially many-to-many and inflate transaction volumes. The join is therefore deliberately protected with `validate='one_to_one'`.


In [ ]:
def key_report(dataset):
    rows=[]
    for year, df in sources[dataset].items():
        rows.append({'dataset':dataset,'year':year,'rows':len(df),'unique_codcom':df.CodCom.nunique(),'missing_codcom':df.CodCom.isna().sum(),'duplicate_codcom':df.CodCom.duplicated().sum()})
    return pd.DataFrame(rows)

quality = pd.concat([key_report(k) for k in sources], ignore_index=True)
quality.sort_values(['dataset','year'])

assert quality.loc[quality.dataset.eq('lista_com'),'duplicate_codcom'].eq(0).all()


## 4. Build the municipality-year analytical spine

`LISTA-COM` supplies the geographic spine. Transaction measures are added with left joins, preserving municipalities listed by OMI even when a category is missing.


In [ ]:
def prepare_year(year):
    base = sources['lista_com'][year].copy()
    geography = [c for c in ['Area','Regione','Provincia','CodCom','Comune','Cap','TAGLIA MERCATO'] if c in base.columns]
    base = base[geography]
    for dataset in ['valori_res','valori_com','valori_per']:
        src = sources[dataset][year].copy()
        base = base.merge(src, on='CodCom', how='left', validate='one_to_one', suffixes=('', f'_{dataset}'))
    base['year'] = year
    return base

transactions_full = pd.concat([prepare_year(y) for y in YEARS], ignore_index=True)
assert not transactions_full.duplicated(['year','CodCom']).any()

# Explicit numeric conversion: zero remains zero; invalid text becomes missing.
id_cols = {'Area','Regione','Provincia','CodCom','Comune','Cap','TAGLIA MERCATO','year'}
for col in [c for c in transactions_full.columns if c not in id_cols]:
    transactions_full[col] = pd.to_numeric(transactions_full[col], errors='coerce')

print(f'Rows: {len(transactions_full):,}')
print(f'Municipality-years: {transactions_full[["year","CodCom"]].drop_duplicates().shape[0]:,}')

## 5. Residential NTN integrity check

The residential file contains total NTN plus five size classes. The size classes should reconcile to total NTN apart from rounding. We **flag** discrepancies rather than silently correcting them.


In [ ]:
size_cols = [c for c in transactions_full.columns if re.search(r'NTN YEAR .*mq', c, re.I)]
total_ntn_col = 'NTN_YEAR' if 'NTN_YEAR' in transactions_full.columns else next((c for c in transactions_full.columns if re.fullmatch(r'NTN_YEAR', c, re.I)), None)
print('Size classes:', size_cols)
print('Total NTN:', total_ntn_col)

if len(size_cols) == 5 and total_ntn_col:
    transactions_full['ntn_size_sum'] = transactions_full[size_cols].sum(axis=1, min_count=1)
    transactions_full['ntn_reconciliation_diff'] = transactions_full['ntn_size_sum'] - transactions_full[total_ntn_col]
    transactions_full['ntn_reconciles'] = transactions_full['ntn_reconciliation_diff'].abs().le(0.05)
    reconciliation = transactions_full['ntn_reconciles'].value_counts(dropna=False).rename_axis('reconciles').to_frame('rows')
    reconciliation['share_pct'] = reconciliation['rows'] / len(transactions_full) * 100
    reconciliation
else:
    print('Schema warning: size-band reconciliation not available.')

## 6. Coverage and national market activity

For national activity, aggregate **sum of NTN**, not the mean across municipalities. This preserves the economic meaning of transaction volume.


In [ ]:
coverage = transactions_full.groupby('year').agg(municipalities=('CodCom','nunique'), provinces=('Provincia','nunique'), regions=('Regione','nunique')).reset_index()
national_ntn = transactions_full.groupby('year', as_index=False)[total_ntn_col].sum(min_count=1).rename(columns={total_ntn_col:'national_ntn'}) if total_ntn_col else pd.DataFrame()
if not national_ntn.empty:
    national_ntn['yoy_pct'] = national_ntn.national_ntn.pct_change() * 100

display(coverage)
display(national_ntn)

In [ ]:
if not national_ntn.empty:
    plt.figure(figsize=(10,5))
    plt.plot(national_ntn.year, national_ntn.national_ntn, marker='o')
    plt.title('Italy — residential normalized transactions (NTN)')
    plt.xlabel('Year')
    plt.ylabel('NTN')
    plt.grid(alpha=0.25)
    plt.show()


## 7. Regional concentration

Regional rankings use total NTN by region. This avoids overweighting small municipalities.


In [ ]:
regional_ntn = (transactions_full.groupby(['year','Regione'], as_index=False)[total_ntn_col].sum(min_count=1).rename(columns={total_ntn_col:'ntn'})) if total_ntn_col else pd.DataFrame()
if not regional_ntn.empty:
    latest_year = regional_ntn.year.max()
    latest = regional_ntn[regional_ntn.year.eq(latest_year)].sort_values('ntn', ascending=False)
    latest.head(10)
    plt.figure(figsize=(10,6))
    top = latest.head(10).sort_values('ntn')
    plt.barh(top.Regione, top.ntn)
    plt.title(f'Top regions by residential NTN — {latest_year}')
    plt.xlabel('NTN')
    plt.grid(axis='x', alpha=0.25)
    plt.show()

## 8. Residential transaction mix by size

Shares show whether the structure of residential demand changes independently of overall market size.


In [ ]:
if len(size_cols) == 5:
    labels = ['≤50 m²','50–85 m²','85–115 m²','115–145 m²','>145 m²']
    size_long = transactions_full.melt(id_vars=['year','CodCom','Regione','Comune'], value_vars=size_cols, var_name='size_band', value_name='ntn')
    mapping = dict(zip(size_cols, labels))
    size_long['size_band'] = size_long.size_band.map(mapping)
    size_mix = size_long.groupby(['year','size_band'], as_index=False).ntn.sum(min_count=1)
    size_mix['share_pct'] = size_mix.ntn / size_mix.groupby('year').ntn.transform('sum') * 100
    display(size_mix)
    pivot = size_mix.pivot(index='year', columns='size_band', values='share_pct')
    ax = pivot.plot(figsize=(10,6), marker='o')
    ax.set_title('Residential transaction mix by property size')
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of NTN (%)')
    ax.grid(alpha=0.25)
    plt.show()

## 9. Municipality panel

The municipality-year panel is the main analytical hand-off: it can support rankings, segmentation and the future price–volume model.


In [ ]:
municipality_panel = transactions_full[['year','CodCom','Comune','Regione','Provincia','TAGLIA MERCATO',total_ntn_col]].rename(columns={total_ntn_col:'ntn'}).copy()
municipality_panel = municipality_panel.sort_values(['CodCom','year'])
municipality_panel['ntn_yoy_pct'] = municipality_panel.groupby('CodCom').ntn.pct_change() * 100
latest_panel = municipality_panel[municipality_panel.year.eq(municipality_panel.year.max())].copy()
latest_panel.sort_values('ntn', ascending=False).head(15)

## 10. Data-quality dashboard and hand-off

The final checks make the notebook auditable and prevent silent structural errors.

### Controls
- annual release inventory is discovered automatically;
- `CodCom` duplicates are checked before joins;
- joins enforce one-to-one relationships;
- municipality-year uniqueness is asserted;
- residential size classes are reconciled with total NTN when the schema permits;
- missing values are preserved rather than imputed.

### Interpretation limits
NTN is a normalized transaction indicator, not a simple count of deeds or physical properties. Volume trends should not be interpreted as price trends. Geographic coverage and reference structures may also change over time.

### Next step
Link this validated municipality-year transaction panel to the quotation dataset from Notebook 01 using a controlled geographic/reference-period key. The target analytical dataset will contain **OMI price midpoint + residential NTN**, enabling a genuine price–volume market analysis.